In [12]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
pedestrians = cv2.VideoCapture('assignment-3-data/pedestrians.avi')
ret, frame1 = pedestrians.read()
print("Frame 1 shape:", frame1.shape)
print("Frame 1 data type:", frame1.dtype)

In [ ]:
prev = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

ret, frame2 = pedestrians.read()
next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(prev, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)

print("Flow shape:", flow.shape)
print("Flow data type:", flow.dtype)

magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)
print("Magnitude min:", magnitude.min(), "max:", magnitude.max())


In [ ]:
hsv = np.zeros((576, 768, 3), dtype=np.uint8)

hsv[..., 0] = angle / 2
hsv[..., 1] = 255
hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)

flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

plt.figure()
plt.imshow(flow_rgb)
plt.title("Optical flow (color = direction, brightness = speed)")
plt.savefig("assignment-3-data/outputs_2/flow_color_frame.png")
plt.show()



In [13]:
pedestrians.set(cv2.CAP_PROP_POS_FRAMES, 0)

ret, frame = pedestrians.read()
prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

os.makedirs("assignment-3-data/outputs_2", exist_ok=True) 
fps = pedestrians.get(cv2.CAP_PROP_FPS) 
fourcc = cv2.VideoWriter_fourcc(*"XVID") 
flow_video = cv2.VideoWriter("assignment-3-data/outputs_2/flow_color.avi", fourcc, fps, (768, 576)) 

while True:
    ret, frame = pedestrians.read()
    if not ret:
        break
    next = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prev, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)

    hsv = np.zeros((576, 768, 3), dtype=np.uint8)
    hsv[..., 0] = angle / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    flow_video.write(flow_bgr)

    prev = next
flow_video.release() 

In [16]:
def filter_flow_by_direction(flow, mag_threshold = 1.0):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)

    moving_right = (angle < 30) | (angle > 330)
    moving_enough = magnitude > mag_threshold
    mask = moving_right & moving_enough

    binary = mask.astype(np.uint8) * 255
    return binary
    
    

In [ ]:
binary = filter_flow_by_direction(flow, mag_threshold=1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(flow_rgb)
axes[0].set_title("Optical flow (color)")
axes[1].imshow(binary, cmap='gray')
axes[1].set_title("Moving right (binary)")
plt.savefig("assignment-3-data/outputs_2/moving_right_frame.png")
plt.show()

In [ ]:
pedestrians.set(cv2.CAP_PROP_POS_FRAMES, 0)

ret, frame = pedestrians.read()
prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

fps = pedestrians.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*"XVID")
binary_video = cv2.VideoWriter("assignment-3-data/outputs_2/moving_right.avi", fourcc, fps, (768, 576), isColor=False)  

while True:
    ret, frame = pedestrians.read()
    if not ret:
        break
    next = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prev, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)

    binary = filter_flow_by_direction(flow, mag_threshold=1.0)  
    binary_video.write(binary)  

    prev = next

binary_video.release()